Síntesis Geográfica por Inferencia Neuronal

Loading the drive

In [1]:
from google.colab import drive
import os

# Montar el drive
drive.mount('/content/drive')

Mounted at /content/drive


Inferencia y Generación de Mapa (.tif)

In [ ]:
import torch
import torch.nn as nn
import numpy as np
import rasterio
from tqdm import tqdm
import os

# --- 1. Definición de la Arquitectura (Requerida para la Inferencia) ---
class UNet(nn.Module):
    def __init__(self):
        super(UNet, self).__init__()
        def conv_block(in_ch, out_ch):
            return nn.Sequential(
                nn.Conv2d(in_ch, out_ch, kernel_size=3, padding=1),
                nn.BatchNorm2d(out_ch),
                nn.ReLU(inplace=True),
                nn.Conv2d(out_ch, out_ch, kernel_size=3, padding=1),
                nn.BatchNorm2d(out_ch),
                nn.ReLU(inplace=True)
            )
        self.enc1, self.enc2, self.enc3 = conv_block(1, 64), conv_block(64, 128), conv_block(128, 256)
        self.pool = nn.MaxPool2d(2)
        self.bottleneck = conv_block(256, 512)
        self.up3 = nn.ConvTranspose2d(512, 256, 2, 2)
        self.dec3 = conv_block(512, 256)
        self.up2 = nn.ConvTranspose2d(256, 128, 2, 2)
        self.dec2 = conv_block(256, 128)
        self.up1 = nn.ConvTranspose2d(128, 64, 2, 2)
        self.dec1 = conv_block(128, 64)
        self.final_conv = nn.Conv2d(64, 1, kernel_size=1)

    def forward(self, x):
        s1 = self.enc1(x); p1 = self.pool(s1)
        s2 = self.enc2(p1); p2 = self.pool(s2)
        s3 = self.enc3(p2); p3 = self.pool(s3)
        b = self.bottleneck(p3)
        d3 = self.dec3(torch.cat((self.up3(b), s3), dim=1))
        d2 = self.dec2(torch.cat((self.up2(d3), s2), dim=1))
        d1 = self.dec1(torch.cat((self.up1(d2), s1), dim=1))
        return torch.sigmoid(self.final_conv(d1))

# --- 2. Configuración de Rutas ---
path_model = "/content/drive/MyDrive/DOCTORADO 2026/VISION ARTIFICIAL/E5/Models/Weights/best_model.pth"
path_cem = "/content/drive/MyDrive/DOCTORADO 2026/VISION ARTIFICIAL/E5/Datasets/Train/CEM_Normalized.npy"
path_output = "/content/drive/MyDrive/DOCTORADO 2026/VISION ARTIFICIAL/E5/Results/Predictions/Prediccion_Riesgo_Veracruz.tif"
path_ref_tif = "/content/drive/MyDrive/DOCTORADO 2026/VISION ARTIFICIAL/E4/Input data/2. Mapa Veracruz georeferenciado.tif"

def ejecutar_inferencia():
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

    # Cargar Modelo
    model = UNet().to(device)
    if os.path.exists(path_model):
        model.load_state_dict(torch.load(path_model, map_location=device))
        model.eval()
    else:
        print("Error: No se encontró el archivo de pesos (.pth)"); return

    # Cargar Datos
    cem = np.load(path_cem)
    h, w = cem.shape
    prediction_map = np.zeros((h, w), dtype=np.float32)
    patch_size = 512

    print(f"Iniciando síntesis en {device}...")
    with torch.no_grad():
        for i in tqdm(range(0, h, patch_size)):
            for j in range(0, w, patch_size):
                i_end, j_end = min(i + patch_size, h), min(j + patch_size, w)
                tile = np.zeros((patch_size, patch_size), dtype=np.float32)
                tile[:i_end-i, :j_end-j] = cem[i:i_end, j:j_end]

                input_t = torch.from_numpy(tile).unsqueeze(0).unsqueeze(0).to(device)
                pred = model(input_t).squeeze().cpu().numpy()
                prediction_map[i:i_end, j:j_end] = pred[:i_end-i, :j_end-j]

    # Guardar GeoTIFF
    with rasterio.open(path_ref_tif) as src:
        meta = src.meta.copy()
        meta.update(dtype=rasterio.float32, count=1)

    with rasterio.open(path_output, 'w', **meta) as dst:
        dst.write(prediction_map, 1)

    print(f"\nMapa generado exitosamente: {path_output}")

if __name__ == "__main__":
    ejecutar_inferencia()

Iniciando síntesis en cpu...


100%|██████████| 17/17 [44:18<00:00, 156.40s/it]



Mapa generado exitosamente: /content/drive/MyDrive/DOCTORADO 2026/VISION ARTIFICIAL/E5/Results/Predictions/Prediccion_Riesgo_Veracruz.tif
